In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import sqlite3

# Create a local SQLite database (no server needed)
conn = sqlite3.connect('analysis.db')

# Create and load a table
employees = pd.DataFrame({
    'emp_id':     [1, 2, 3, 4, 5],
    'Name':       ['Ama', 'Kofi', 'Abena', 'Kwame', 'Efua'],
    'Department': ['Sales', 'Tech', 'Sales', 'HR', 'Tech'],
    'Salary':     [3200, 5800, 3900, 4500, 5100]
})

employees.to_sql('employees', conn, if_exists='replace', index=False)
print("Database created and table loaded!")

Database created and table loaded!


In [2]:
# Read from database into pandas
df = pd.read_sql("SELECT * FROM employees", conn)
df

,emp_id,Name,Department,Salary
0,1,Ama,Sales,3200
1,2,Kofi,Tech,5800
2,3,Abena,Sales,3900
3,4,Kwame,HR,4500
4,5,Efua,Tech,5100


In [3]:
df = pd.read_sql("SELECT * FROM employees WHERE Salary > 4000", conn)
df

,emp_id,Name,Department,Salary
0,2,Kofi,Tech,5800
1,4,Kwame,HR,4500
2,5,Efua,Tech,5100


In [4]:
df = pd.read_sql("SELECT Department, AVG(Salary) as avg_salary FROM employees GROUP BY Department", conn)
df

,Department,avg_salary
0,HR,4500.0
1,Sales,3550.0
2,Tech,5450.0


In [5]:
df = pd.read_sql("SELECT * FROM employees ORDER BY Salary DESC", conn)
df

,emp_id,Name,Department,Salary
0,2,Kofi,Tech,5800
1,5,Efua,Tech,5100
2,4,Kwame,HR,4500
3,3,Abena,Sales,3900
4,1,Ama,Sales,3200


In [6]:
# Pull data with SQL, then analyze with pandas
df = pd.read_sql("SELECT * FROM employees WHERE Salary > 3000", conn)

# Now use pandas on top
result = df.groupby('Department')['Salary'].agg(['mean', 'max', 'count'])
print(result)

              mean   max  count
Department                     
HR          4500.0  4500      1
Sales       3550.0  3900      2
Tech        5450.0  5800      2


In [7]:
# Create a summary and save it back to the database
summary = df.groupby('Department')['Salary'].agg(['mean', 'max', 'count']).reset_index()
summary.columns = ['Department', 'Avg_Salary', 'Max_Salary', 'Employee_Count']

summary.to_sql('dept_summary', conn, if_exists='replace', index=False)

# Verify it saved
pd.read_sql("SELECT * FROM dept_summary", conn)

,Department,Avg_Salary,Max_Salary,Employee_Count
0,HR,4500.0,4500,1
1,Sales,3550.0,3900,2
2,Tech,5450.0,5800,2
